In [ ]:
%pip install pandas numpy seaborn matplotlib scipy pydeseq2 gprofiler adjustText
!rm -drf Data
!mkdir Data
!wget https://raw.githubusercontent.com/simonelsasser/SP1-01-Ex/f048b4557e63417354839094bf812fd808a67dde/Kumar_NCB/Data/metaData.csv -O Data/metaData.csv
!wget https://raw.githubusercontent.com/simonelsasser/SP1-01-Ex/f048b4557e63417354839094bf812fd808a67dde/Kumar_NCB/Data/rsem.merged.gene_counts_2021.tsv -O Data/rsem.merged.gene_counts_2021.tsv
!wget https://raw.githubusercontent.com/simonelsasser/SP1-01-Ex/f048b4557e63417354839094bf812fd808a67dde/Kumar_NCB/Data/gene_group_mapping.csv -O Data/gene_group_mapping.csv
!ls Data/
!cat Data/metaData.csv

In [ ]:
import pandas as pd
import numpy as np

#########################
######   TASK 1   #######
#      read in data     #
#########################

file_path = "Data/rsem.merged.gene_counts_2021.tsv"
# Read in count file "Data/rsem.merged.gene_counts_2021.tsv", it is a tab-separated file with the first column as index
count_table = pd.read_csv()
# Print the first 5 rows of the count table
print(count_table.head())

In [ ]:

# we are going to remove 'transcript_id(s)' from the count table, as they are not needed for the analysis
count_table = count_table.drop('transcript_id(s)', axis=1)
# Print the head again
print(count_table.head())


In [ ]:
#Check the numbers in the count table - it looks like they are all intergers. Is this what you expect?

#Let's take the first column (Naive_EZH2i_R1)- What is the min, max, mean, median and variance of the counts in the count table? 
print("Min:", count_table.iloc[:, 0].min())

# Are there any zeros? How many genes in total and how many genes have zero counts in all samples?
print("Total number of genes:", count_table.shape[0])

In [ ]:

# What is the distribution of counts? Can you make a graph to inspect the distribution? 
import matplotlib.pyplot as plt

# Plot a histogram of the counts in the first column, limit the x axis to 0-100 and set the bin width to 10
# What do you observe? What does the high peak at zero and long tail mean?
plt.hist()
plt.xlabel('Counts')
plt.ylabel('Frequency')
plt.title('Distribution of Counts')
plt.xlim(0, 100)
plt.ylim(0, 2000)
plt.show()

In [ ]:
# Let's try to log transform the counts with log10 (add + 1 to the count table to avoid log(0)!)
plt.hist(, bins=100)
plt.xlabel('Log10(Counts + 1)')
plt.ylabel('Frequency')
plt.title('Distribution of Counts')
plt.ylim(0, 1000)
plt.show()

In [ ]:
# For comparing gene expression, we need to understand the distribution of read counts not within one sample, but for each gene across all samples. Let's plot the distribution of counts the pluripotency factor gene NANOG and GAPDH (housekeeping gene)across all samples. What do you observe?
print(count_table.loc['NANOG', :])

In [ ]:
# Now something looks strange - the counts for "Primed" appear to be much higher even for housekeeping gene GAPDH. Let's plot the distribution of counts for all samples and see if we can spot any batch effect or other technical bias. We can use a box plot for this.
import seaborn as sns
# Convert the count table to long format for seaborn box plot and log transform
count_table_long = count_table.melt(var_name='Sample', value_name='Count')
count_table_long['Count'] = np.log10(count_table_long['Count'] + 1)
# Plot a box plot of counts for each sample
plt.figure(figsize=(10, 6))
sns.boxplot(x='Sample', y='Count', data=count_table_long)
plt.yscale('log')
plt.xlabel('Sample')
plt.ylabel('Counts (log scale)')
plt.title('Distribution of Counts Across Samples')
plt.xticks(rotation=90)
plt.show()

In [ ]:

# Since we learned that boxplots could hide the distribution of the data, let's try a violin plot instead
# Plot a violin plot of counts for each sample

In [ ]:
# Clearly there is a big difference between the "Naive" and "Primed" samples
# A trivial source of this difference is that libraries are sequenced to different depths, this happens because it is difficult to exactly normalize the NGS library
# before sequencing
# In the current dataset, the BGI generated almost exactly the same sequencing depth. But we actually know why there is a sytematic different:
# The "Naive" cells are grown on mouse feader cells, so in the RNA extraction
# we have a big contribution of mouse RNA, this will be part of the library but it will not map to the human genome. So a lot of the reads in the library are 'lost' to mouse RNA,
# and as a result we appear to have less total reads.

# In order to compare gene expression between the "Naive" and "Primed" samples, we need to normalize the counts to account for this difference in sequencing depth. One common way to do this is to calculate counts per million (CPM) or transcripts per million (TPM). Let's calculate CPM for each sample and plot the distribution again.
# Calculate CPM: Normalize each row count by the total read count of the sample (column sum) and multiply by 1 million
cpm_table = count_table.div(count_table.sum(axis=0), axis=1) * 1e6
# now check NANOG and GAPDH again

In [ ]:

# Let's plot the CPM valus after log transformation
# Convert to long format for plotting and log transform 
cpm_table_long = cpm_table.melt(var_name='Sample', value_name='CPM')
cpm_table_long['CPM'] = np.log10(cpm_table_long['CPM'] + 1)
# Plot a violin plot of CPM for each sample
plt.figure(figsize=(10, 6))
sns.violinplot()
plt.xlabel('Sample')
plt.ylabel('CPM (log scale)')
plt.title('Distribution of CPM Across Samples')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Great, now the "Naive" and "Primed" samples look much more similar, we can now compare gene expression between them!
### WARNING: CPM is not a good normalization method for differential expression analysis, because it is highly skewed by the largest counts in the table (the tail)
# Better would be to align the distributions, e.g. by minimizing the variance of the log fold changes between samples, this is what DESeq2 does with its size factor estimation. We will learn about this in the next session.
# But you can see that for now CPM works quite OK for the given data

In [ ]:
## DESeq2 works under the assumption of a negative binomial distribution, which means that the variance should be higher than the mean. 
# This is an empirical observation that holds for most genes in RNA-seq data, and it is one of the reasons why DESeq2 uses a negative binomial distribution to model the counts.
# The alternative model wpuld be the Poisson distribution, which assumes that the variance is equal to the mean. 
# Let's check if this is true for our data by plotting the mean-variance relationship of the CPM values.
means = cpm_table.mean(axis=1)
vars_ = cpm_table.var(axis=1)

plt.scatter(means, vars_, s=5, alpha=0.3)
plt.xscale("log")
plt.yscale("log")

plt.xlabel("Mean count")
plt.ylabel("Variance")
plt.title("Mean-variance relationship")
x = np.logspace(-3, 5, 100)
plt.plot(x, x, color="red", label="Poisson variance")
plt.show()


In [ ]:
# Read in file 
experiment_design = pd.read_csv("Data/metaData.csv", sep=",")

# Get an overview of the file
print(experiment_design)

In [ ]:
#########################
######   TASK 2   #######
# Preparation for DESeq #
#########################

# prepare metaData
# shape meta for DESeq analysis (name as index, treatment and condition as column)
meta = experiment_design.set_index("Name")
#meta = meta[["treatment","condition"]]
# shape meta for DESeq analysis (name as index, treatment and condition as column)
# add a column "design" that is a combination of treatment and condition, e.g. "Naive_EZH2i" and "Primed_EZH2i"
meta["design"] = meta["condition"] + "_" + meta["treatment"]
print(meta)

In [ ]:
# prepare count table 
# filter for lowly expressed genes
genes_to_keep = count_table.columns[count_table.sum(axis=0) >= 10]
counts_df = count_table[genes_to_keep]

# convert dataframe to integer
counts_df = counts_df.astype(int)

# transpose dataframe for Deseq analysis
counts_df = counts_df.T

# tricky part: make sure that the order of samples in the count table matches the order of samples in the meta data, otherwise DESeq2 will throw an error. 
# We can do this by reindexing the count table with the index of the meta data.
print(counts_df.index)
counts_df = counts_df.loc[meta.index]
print(counts_df.index)

In [ ]:
#######################
######   TASK 3   #######
#     Running DESeq2    #
#########################

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# create DeseqDataSet
dds = DeseqDataSet(
    counts=,
    metadata=,
    design=,
    refit_cooks=True)


In [ ]:
# run dds.deseq2()
dds.deseq2()

In [ ]:

#extract and log transform the normalized counts for PCA, it is stored in dds.layers["normed_counts"], a samples x genes matrix
lnc = np.log1p(dds.layers["normed_counts"])  # samples x genes
 
# Basic PCA with scikit-learn
from importlib.metadata import metadata

from sklearn.decomposition import PCA
 
X = np.asarray(lnc)                         # make sure it's a numeric array
pca = PCA(n_components=2).fit_transform(X)  # PCs across samples
 
# Plot, color by design, provide legend and labels of points
plt.figure(figsize=(8, 6))
for design in meta["design"].unique():
    mask = meta["design"] == design
    plt.scatter(pca[mask, 0], pca[mask, 1], label=design, marker="o", s=100)

plt.xlabel("PC1 - primed <-> naive")
plt.ylabel("PC2 - none <-> EZH2i")
plt.title("PCA by condition")
plt.legend()
plt.show()


In [ ]:
# now print the normalized read counts for GAPDH and NANOG from DESeq2, which are stored in the "normalized_counts" layer of the dds object.
dds.layers.keys()  # check what layers are available in the dds object
normed_counts = pd.DataFrame(dds.layers["normed_counts"], index=counts_df.index, columns=counts_df.columns)
print("GAPDH normalized read counts:", normed_counts.loc[:, "GAPDH"])
print("NANOG normalized read counts:", normed_counts.loc[:, "NANOG"])

In [ ]:
# let's make a nicer bar plot of the expressin values per design for GAPDH and NANOG
# we need to annotate the rows to group them by design, we can do this by melting the dataframe and then merging with the meta data
# melt, keep the Name and gene_identifier as columns, and the gene columns as values
normed_counts_long = normed_counts.reset_index().melt(id_vars=["Name"], var_name="gene_id", value_name="normalized_count")
normed_counts_long["design"] = normed_counts_long["Name"].map(meta["design"])
print(normed_counts_long.head())

In [ ]:

# Plot normalized counts for GAPDH, use bar for mean and error bar for standard deviation across samples of the same design, plot jitter of individual samples on top of the bar plot
plt.figure(figsize=(10, 6))
sns.barplot(data=normed_counts_long[normed_counts_long["gene_id"].isin(["GAPDH"])], x="design", y="normalized_count", ci="sd")
sns.stripplot(data=normed_counts_long[normed_counts_long["gene_id"].isin(["GAPDH"])], x="design", y="normalized_count", color="black", alpha=0.5, jitter=True)
plt.xlabel("Design")
plt.ylabel("Normalized Count")
plt.title("Normalized Counts for GAPDH and NANOG")
plt.xticks(rotation=45)
plt.show()

# Plot normalized counts for NANOG, use bar for mean and error bar for standard deviation across samples of the same design, plot jitter of individual samples on top of the bar plot


# Plot normalized counts for CD24, use bar for mean and error bar for standard deviation across samples of the same design, plot jitter of individual samples on top of the bar plot


In [ ]:
#########################
######   TASK 4   #######
#    Heatmap Figure 2e  #
#########################

# Check out Figure 2e, the figure shows a selection of differentially expressed genes between "Naive" and "Primed" samples,
# using log TPM values

![Figure 2](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41556-022-00916-w/MediaObjects/41556_2022_916_Fig2_HTML.png)

In [ ]:
#########################
######   TASK 4   #######
#    Heatmap Figure 2e  #
#########################

genes_2e=["KLF17","HORMAD1","TRIM6O","ZNF729","ALPP","TBX3","IL6ST","KLF5","KLF4","TFAP2C","DPPA3","ALPPL2","DPPAS","DNMT3L","KHDC3L","CD24","THY1","OTX2","SFRP2","ZIC2","PTPRZ1","DUSP6","CYTL1","HMX2","POU5F1","SOX2","UTF1","NANOG"]

#from normed_counts_long, subset for the genes in genes_2e and plot a heatmap of the normalized counts for these genes across the samples, group by design
normed_counts_log = normed_counts.reset_index().melt(id_vars=["Name"], var_name="gene_id", value_name="normalized_count")
normed_counts_log["normalized_count"] = np.log2(normed_counts_log["normalized_count"]+1)
normed_counts_log["design"] = normed_counts_log["Name"].map(meta["design"])
normed_counts_2e = normed_counts_log[normed_counts_log["gene_id"].isin(genes_2e)]
normed_counts_2e_pivot = normed_counts_2e.pivot(index="gene_id", columns="Name", values="normalized_count")
# reindex to maintain the original order from genes_2e
normed_counts_2e_pivot = normed_counts_2e_pivot.reindex(genes_2e)
# sort columns by design order: naive_none, naive_EZH2i, primed_none, primed_EZH2i
design_order = ["naive_none","naive_EZH2i","primed_none","primed_EZH2i"]
sorted_columns = sorted(normed_counts_2e_pivot.columns, 
                       key=lambda x: (design_order.index(meta.loc[x, "design"]), x))
normed_counts_2e_pivot = normed_counts_2e_pivot[sorted_columns]
# plot a heatmap of the normalized counts for these genes across the samples, group by design
plt.figure(figsize=(4, 8))
sns.heatmap(normed_counts_2e_pivot, cmap="viridis", yticklabels=True)
plt.title("Heatmap of Normalized Counts for Figure 2e Genes")
plt.xlabel("Sample")
plt.ylabel("Gene")
plt.xticks(rotation=90)
plt.show()


In [ ]:
#########################
######   TASK 5   #######
#    volcanoe plots     #
#########################

# Extract the results of the DESeq2 analysis for the contrast "Naive_none" vs "Primed_none", which is then
# stored in the "results_df" attribute of the ds object the contrast is defined as:
# contrast=["design", "naive_none", "primed_none"],

ds = DeseqStats(dds, n_cpus=1)
ds.summary()
naive_vs_primed_raw_lfc = ds.results_df["log2FoldChange"]
ds.lfc_shrink(coeff="design[T.naive_none]")

naive_vs_primed = ds.results_df
ds.summary()

In [ ]:
########
# Plot a so-called MA-plot: 
# xaxis is the expression of the gene (log10[baseMean])
# yaxis is the log2 fold change, color points with padj < 0.05 in red and the rest in black

plt.figure(figsize=(8, 6))
colors = np.where(naive_vs_primed["padj"] < 0.01, "red", "black")
plt.scatter()
plt.xlabel("Log10 Base Mean")
plt.ylabel("Log2 Fold Change (uncorrected/without lfc shrinkage)")
plt.title("MA Plot of DESeq2 Results")
plt.show()

In [ ]:
########
# looks like there are very high fold-changes for very lowly expressed genes, 
# this is a common problem in DE analysis, because the fold change is not well defined for low counts.
# Thats why we use a method called "lfcShrink" that shrinks the log fold for small counts


plt.figure(figsize=(8, 6))
colors = np.where(naive_vs_primed["padj"] < 0.01, "red", "black")
plt.scatter()
plt.xlabel("Log10 Base Mean")
plt.ylabel("Log2 Fold Change")
plt.title("MA Plot of DESeq2 Results")
plt.show()

In [ ]:
#######
# Volcano plot:
# xaxis is the log2 fold change, yaxis is the -log10 of the adjusted p-value

#print Volcano plot plot of the results, color points with padj < 0.05 and |lfc|>1 in red and the rest in black
plt.figure(figsize=(10, 7))
colors = np.where((naive_vs_primed["padj"] < 0.01) & (naive_vs_primed["log2FoldChange"] > 1), "blue", np.where((naive_vs_primed["padj"] < 0.01) & (naive_vs_primed["log2FoldChange"] < -1), "red", "black"))
plt.scatter(naive_vs_primed_raw_lfc, -np.log10(naive_vs_primed["padj"]), c=colors, alpha=0.5, s=4)
plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10 Adjusted P-value")
plt.title("Volcano Plot of DESeq2 Results")

# Add text labels for genes in genes_2e
texts = []
for gene in genes_2e:
    if gene in naive_vs_primed.index:
        x = naive_vs_primed.loc[gene, "log2FoldChange"]
        y = -np.log10(naive_vs_primed.loc[gene, "padj"])
        texts.append(plt.text(x, y, gene, fontsize=8))
plt.show()


### Remember:

| Measuremen      | Meaning                                |
| --------------- | ---------------------------------------|
| p-value         | evidence against null                  |
| padj-valu       | p after correction for multiple testing|
| log2FC          | estimated effect size                  |
| shrunken log2FC | stabilized plausible effect size       |


In [ ]:
### Now lets look at Figure 2a! comparing EZH2i vs none in the "Naive" condition, extract the results for this contrast and plot a volcano plot as well.

![Figure 3](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41556-022-00916-w/MediaObjects/41556_2022_916_Fig3_HTML.png)

In [ ]:
#########################
######   TASK    #######
#    volcanoe plots     #
#########################

# Extract the results of the DESeq2 analysis for the contrast "Naive_EZH2i" vs "Naive_none", which is then
# stored in the "results_df" attribute of the ds object

ds = DeseqStats()
ds.summary()
EZH2i_vs_naive_raw_lfc = ds.results_df["log2FoldChange"]
ds.lfc_shrink(coeff="design[T.naive_none]")

EZH2i_vs_naive = ds.results_df
ds.summary()


In [ ]:
# genes to mark as in volcanoe plot
genes_3a =  ["KRT18",  "H19", "IGF2", "EPAS2", "COL6A3", "MME",
    "VIM", "MMP2", "EGFL6", "CCKBR", "GATA2", "COL15A1", "CD99","EPAS1",
    "LAMB1", "THBD", "FRZB",  "RELN", "TDGF1", "MT1G","DDPA5","NODAL","NANOG","UTF1",
    "FGF4"]


#######
# Volcano plot:
# xaxis is the log2 fold change, yaxis is the -log10 of the adjusted p-value

#print Volcano plot plot of the results, color points with padj < 0.05 and |lfc|>1 in red and the rest in black
plt.figure(figsize=(10, 7))

plt.show()

In [ ]:
#######
# GENE SET ENRICHMENT ANALYSIS
# get naive- and primed-specific genes-sets 

sigUp = EZH2i_vs_naive[(EZH2i_vs_naive["padj"] < 0.01) & (EZH2i_vs_naive["log2FoldChange"] > 2)]

# get significantly downregulated genes
sigDown = EZH2i_vs_naive[(EZH2i_vs_naive["padj"] < 0.01) & (EZH2i_vs_naive["log2FoldChange"] < -2)]

# perform gene set enrichment analysis with gprofiler for the upregulated genes, use the "gost" function and specify the organism as "hsapiens"
from gprofiler.gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)

enrichment_up = gp.profile(query=sigUp.index.tolist(), organism="hsapiens")
enrichment_down = gp.profile(query=sigDown.index.tolist(), organism="hsapiens")

###
print(enrichment_up.head())

In [ ]:
print(enrichment_down.head())


In [ ]:

topEZH2i = EZH2i_vs_naive[(EZH2i_vs_naive["padj"] < 0.01) & (EZH2i_vs_naive["log2FoldChange"] < -5)]

#from normed_counts_long, subset for the genes in genes_2e and plot a heatmap of the normalized counts for these genes across the samples, group by design
normed_counts_3a = normed_counts_log[normed_counts_log["gene_id"].isin(topEZH2i.index.tolist())]
normed_counts_3a_pivot = normed_counts_3a.pivot(index="gene_id", columns="Name", values="normalized_count")

# sort columns by design order: naive_none, naive_EZH2i, primed_none, primed_EZH2i
design_order = ["naive_none","naive_EZH2i","primed_none","primed_EZH2i"]
sorted_columns = sorted(normed_counts_3a_pivot.columns, 
                       key=lambda x: (design_order.index(meta.loc[x, "design"]), x))
normed_counts_3a_pivot = normed_counts_3a_pivot[sorted_columns]
# plot a heatmap of the normalized counts for these genes across the samples, group by design
plt.figure(figsize=(4, 8))
sns.heatmap(normed_counts_3a_pivot, cmap="viridis", yticklabels=True)
plt.title("Heatmap of Normalized Counts for Genes most significantly upregulated in EZH2i vs naive")
plt.xlabel("Sample")
plt.ylabel("Gene")
plt.xticks(rotation=90)
plt.show()